# 02 — Data Preprocessing

## Smart India Real Estate Analytics

### Objective

Prepare the cleaned dataset for XGBoost regression while preventing data leakage.

### Workflow

Clean Dataset
→ Feature Selection
→ Train/Test Split
→ Fit Preprocessing on Training Data Only
→ Transform Training and Test Data

### Important Rule

The test set must not influence the fitting of the preprocessing pipeline.

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/processed/cleaned_real_estate.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head(3))

Dataset shape: (14517, 10)


,Name,Property Title,Price,Location,Total_Area,Price_per_SQFT,Description,Baths,Balcony,Price_numeric
0,Casagrand ECR 14,"4 BHK Flat for sale in Kanathur Reddikuppam, C...",₹1.99 Cr,"Kanathur Reddikuppam, Chennai",2583,7700.0,Best 4 BHK Apartment for modern-day lifestyle ...,4,Yes,19900000.0
1,"Ramanathan Nagar, Pozhichalur,Chennai",10 BHK Independent House for sale in Pozhichal...,₹2.25 Cr,"Ramanathan Nagar, Pozhichalur,Chennai",7000,3210.0,Looking for a 10 BHK Independent House for sal...,6,Yes,22500000.0
2,DAC Prapthi,"3 BHK Flat for sale in West Tambaram, Chennai",₹1.0 Cr,"Kasthuribai Nagar, West Tambaram,Chennai",1320,7580.0,"Property for sale in Tambaram, Chennai. This 3...",3,No,10000000.0


In [9]:
import re

def extract_bhk(title):
    title = str(title).lower()

    # Handle BHK values such as 1 BHK, 2 BHK, 5+ BHK
    match = re.search(r'(\d+)\s*\+?\s*bhk', title)

    if match:
        return int(match.group(1))

    # Handle RK properties
    match = re.search(r'(\d+)\s*rk', title)

    if match:
        return int(match.group(1))

    return np.nan


def extract_property_type(title):
    title = str(title).lower()

    if "villa" in title:
        return "Villa"
    elif "house" in title:
        return "House"
    elif "flat" in title or "apartment" in title:
        return "Apartment"
    elif "studio" in title:
        return "Studio"
    elif "plot" in title:
        return "Plot"
    else:
        return "Other"


df["BHK"] = df["Property Title"].apply(extract_bhk)
df["Property_Type"] = df["Property Title"].apply(extract_property_type)

print("Feature extraction completed.")
print("BHK missing:", df["BHK"].isna().sum())

print("\nBHK distribution:")
print(df["BHK"].value_counts().sort_index())

print("\nProperty types:")
print(df["Property_Type"].value_counts())

Feature extraction completed.
BHK missing: 21

BHK distribution:
BHK
1.0     3267
2.0     5680
3.0     3127
4.0      989
5.0      618
6.0      296
7.0      143
8.0      132
9.0       78
10.0     166
Name: count, dtype: int64

Property types:
Property_Type
Apartment    9681
House        4101
Villa         735
Name: count, dtype: int64


In [8]:
missing_bhk = df[df["BHK"].isna()]["Property Title"]

print("Examples of missing BHK titles:")
display(missing_bhk.head(20))

Examples of missing BHK titles:


94      1 RK Flat for sale in Oragadam Sriperambattur,...
145           1 RK Flat for sale in Pallikaranai, Chennai
157             1 RK Flat for sale in Manapakkam, Chennai
175                  1 RK Flat for sale in Padur, Chennai
176              1 RK Flat for sale in Senganmal, Chennai
749           1 RK Flat for sale in Moolakazhani, Chennai
779           1 RK Flat for sale in Chettipunyam, Chennai
851     1 RK Independent House for sale in Kosappur, C...
899     1 RK Independent House for sale in Veppampattu...
946            1 RK Flat for sale in Madambakkam, Chennai
954     1 RK Flat for sale in Oragadam Sriperambattur,...
966             1 RK Flat for sale in Triplicane, Chennai
1022          1 RK Villa for sale in Kanchipuram, Chennai
1039           1 RK Flat for sale in Nemilicheri, Chennai
1057    5+ BHK Independent House for sale in Avadi, Ch...
1075         1 RK Flat for sale in West Mambalam, Chennai
1085    1 RK Independent House for sale in Poonamallee...
1118         1

In [10]:
# Select features and target

FEATURES = [
    "Location",
    "Total_Area",
    "Baths",
    "Balcony",
    "BHK",
    "Property_Type"
]

TARGET = "Price_numeric"

X = df[FEATURES].copy()
y = df[TARGET].copy()

print("Selected features:")
print(FEATURES)

print("\nFeature shape:", X.shape)
print("Target shape:", y.shape)

Selected features:
['Location', 'Total_Area', 'Baths', 'Balcony', 'BHK', 'Property_Type']

Feature shape: (14517, 6)
Target shape: (14517,)


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)

Training features: (11613, 6)
Test features: (2904, 6)
Training target: (11613,)
Test target: (2904,)


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

NUMERIC_FEATURES = [
    "Total_Area",
    "Baths",
    "BHK"
]

CATEGORICAL_FEATURES = [
    "Location",
    "Balcony",
    "Property_Type"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, NUMERIC_FEATURES),
    ("cat", categorical_pipeline, CATEGORICAL_FEATURES)
])

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [13]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training processed shape:", X_train_processed.shape)
print("Test processed shape:", X_test_processed.shape)

Training processed shape: (11613, 6026)
Test processed shape: (2904, 6026)


In [14]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

# Save preprocessing pipeline
joblib.dump(
    preprocessor,
    MODEL_DIR / "preprocessor.joblib"
)

# Save processed datasets
joblib.dump(
    X_train_processed,
    MODEL_DIR / "X_train_processed.joblib"
)

joblib.dump(
    X_test_processed,
    MODEL_DIR / "X_test_processed.joblib"
)

# Save targets
joblib.dump(
    y_train,
    MODEL_DIR / "y_train.joblib"
)

joblib.dump(
    y_test,
    MODEL_DIR / "y_test.joblib"
)

print("Preprocessing artifacts saved successfully.")

Preprocessing artifacts saved successfully.
